# 📈 01 — Auto-SARIMA
## Modèle SARIMA avec Grid Search des ordres optimaux

Ce notebook importe les données préparées par `00_Preprocessing.ipynb` et entraîne un SARIMA avec recherche automatique des meilleurs ordres (p,d,q)(P,D,Q,s).

## 1. 📦 Imports & Chargement des données préparées

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("✅ Imports OK")

In [ ]:
# Charger les données préparées par 00_Preprocessing
df = pd.read_csv("prepared_data.csv", index_col=0, parse_dates=True)
train = pd.read_csv("train_data.csv", index_col=0, parse_dates=True)
test = pd.read_csv("test_data.csv", index_col=0, parse_dates=True)

print(f"✅ Données chargées : {len(df)} mois")
print(f"   Train : {len(train)} mois → Test : {len(test)} mois")

## 2. 🔬 Test de Stationnarité ADF

In [ ]:
print("🔬 Test ADF sur la série originale")
adf = adfuller(train["revenue"].dropna())
print(f"   Stat={adf[0]:.3f}, p={adf[1]:.3f}")
print(f"   → {'Stationnaire ✅' if adf[1] < 0.05 else 'Non-stationnaire → différenciation nécessaire'}")

# ACF/PACF plots
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
plot_acf(train["revenue"].diff().dropna(), ax=axes[0], lags=12)
plot_pacf(train["revenue"].diff().dropna(), ax=axes[1], lags=12)
axes[0].set_title("ACF — Série différenciée")
axes[1].set_title("PACF — Série différenciée")
plt.tight_layout()
plt.show()

## 3. ⚙️ Grid Search SARIMA (ordres optimaux)

In [ ]:
print("⏳ Grid Search des ordres SARIMA...")
print("   Recherche sur p∈[0,2], d∈[0,1], q∈[0,2], P∈[0,1], D∈[0,1], Q∈[0,1]")

best_aic, best_order, best_seasonal, best_model = np.inf, None, None, None
S = 12

for p in range(0, 3):
    for d in range(0, 2):
        for q in range(0, 3):
            for P in range(0, 2):
                for D in range(0, 2):
                    for Q in range(0, 2):
                        try:
                            model = SARIMAX(train["revenue"], order=(p,d,q),
                                            seasonal_order=(P,D,Q,S),
                                            enforce_stationarity=False,
                                            enforce_invertibility=False)
                            fitted = model.fit(disp=False, maxiter=200)
                            if fitted.aic < best_aic:
                                best_aic, best_order = fitted.aic, (p,d,q)
                                best_seasonal, best_model = (P,D,Q,S), fitted
                        except:
                            pass

print(f"\n✅ Meilleur SARIMA :")
print(f"   order={best_order}, seasonal_order={best_seasonal}")
print(f"   AIC={best_aic:.2f}")

## 4. 🔮 Prévisions & Évaluation

In [ ]:
# ── Prévisions ────────────────────────────────────────────
sarima_pred = best_model.forecast(steps=12)
sarima_pred.index = test.index
pred_ci = best_model.get_forecast(steps=12).conf_int()
pred_ci.index = test.index

# ── Ljung-Box test ───────────────────────────────────────
lb = acorr_ljungbox(best_model.resid, lags=[6, 12], return_df=True)
print("🔬 Ljung-Box test (résidus) :")
for lag in [6, 12]:
    if lag in lb.index:
        pv = lb.loc[lag, "lb_pvalue"]
        print(f"   Lag {lag}: p={pv:.3f} → {'✅ Résidus blancs' if pv > 0.05 else '⚠️ Résidus corrélés'}")

# ── Métriques ─────────────────────────────────────────────
mae = mean_absolute_error(test["revenue"], sarima_pred)
rmse = np.sqrt(mean_squared_error(test["revenue"], sarima_pred))
mape_val = np.mean(np.abs((test["revenue"].values - sarima_pred.values) / test["revenue"].values)) * 100
r2 = r2_score(test["revenue"], sarima_pred)

print(f"\n📊 SARIMA{best_order}×{best_seasonal}")
print(f"   MAE  : {mae:>12,.0f} €")
print(f"   RMSE : {rmse:>12,.0f} €")
print(f"   MAPE : {mape_val:>11.2f} %")
print(f"   R²   : {r2:>11.4f}")

# ── Graphique ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

ax = axes[0]
train["revenue"].plot(ax=ax, label="Train", color="#1565C0", alpha=0.7, linewidth=2)
test["revenue"].plot(ax=ax, label="Test réel", color="#2E7D32", linewidth=2.5)
sarima_pred.plot(ax=ax, label=f"SARIMA{best_order}×{best_seasonal}", color="#E65100", linestyle="--", linewidth=2.5)
ax.fill_between(pred_ci.index, pred_ci.iloc[:, 0], pred_ci.iloc[:, 1], alpha=0.15, color="#E65100", label="IC 95%")
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2021-06-30"), alpha=0.07, color="red", label="COVID")
ax.set_title(f"SARIMA{best_order}×{best_seasonal} — Prévisions vs Réel", fontsize=13, fontweight="bold")
ax.set_ylabel("Revenue (€)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M€"))
ax.legend(); ax.grid(alpha=0.3)

ax2 = axes[1]
residuals = best_model.resid
ax2.plot(residuals, color="#6A1B9A", alpha=0.8, linewidth=1.5)
ax2.axhline(0, color="red", linestyle="--", linewidth=1.5)
ax2.fill_between(residuals.index, residuals, 0, where=residuals > 0, alpha=0.3, color="green")
ax2.fill_between(residuals.index, residuals, 0, where=residuals < 0, alpha=0.3, color="red")
ax2.set_title("Résidus SARIMA", fontsize=12); ax2.grid(alpha=0.3)

plt.suptitle(f"SARIMA | MAPE: {mape_val:.2f}% | R²: {r2:.4f}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("modele1_sarima.png", dpi=120, bbox_inches="tight")
plt.show()
print("\n💾 Graphique → modele1_sarima.png")

# ── Sauvegarder les prédictions ──────────────────────────
results_df = pd.DataFrame({
    "date": test.index, "reel": test["revenue"], "prediction": sarima_pred,
    "ic_bas": pred_ci.iloc[:, 0], "ic_haut": pred_ci.iloc[:, 1]
})
results_df.to_csv("predictions_sarima.csv", index=False)
print("💾 Prédictions → predictions_sarima.csv")